In [1]:
pip install ultralytics opencv-python numpy

Note: you may need to restart the kernel to use updated packages.


In [2]:
import datetime
import os
import time
import winsound  # Windows साठी built-in sound module
import cv2
from ultralytics import YOLO

# --- १. मॉडेल आणि इनपुट स्त्रोत सेट करा ---
MODEL_PATH = "yolov8n.pt"  # किंवा "pothole_yolov8n.pt"
VIDEO_SOURCE = 0  # वेबकॅमसाठी 0, किंवा व्हिडिओ फाइलचा पाथ

# --- २. स्क्रीनशॉट फोल्डर तयार करा ---
SAVE_DIR = "saved_potholes"
if not os.path.exists(SAVE_DIR):
  os.makedirs(SAVE_DIR)

print("[INFO] Loading YOLO Model...")
model = YOLO(MODEL_PATH)

print("[INFO] Opening Video Stream...")
cap = cv2.VideoCapture(VIDEO_SOURCE)

TARGET_CLASSES = ["pothole", "crack"]

last_alert_time = 0
ALERT_INTERVAL = 2.0  # दर २ सेकंदात जास्तीत जास्त १ सायरन आणि १ स्क्रीनशॉट

# सायरन वाजवण्यासाठी फंक्शन (Frequency: 2500Hz, Duration: 500ms)
def play_siren():
  winsound.Beep(2500, 500)  # frequency, duration in ms


print("[INFO] System Active. Press 'q' to quit.")

while cap.isOpened():
  ret, frame = cap.read()
  if not ret:
    break

  # YOLO Inference
  results = model(frame, conf=0.35, verbose=False)

  pothole_detected = False
  pothole_count = 0

  for result in results:
    boxes = result.boxes
    for box in boxes:
      cls_id = int(box.cls[0])
      class_name = model.names[cls_id]
      conf = float(box.conf[0])

      if class_name.lower() in TARGET_CLASSES:
        pothole_detected = True
        pothole_count += 1
        x1, y1, x2, y2 = map(int, box.xyxy[0])

        # Bounding Box
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 2)
        label = f"{class_name} {conf:.2f}"
        cv2.putText(
            frame,
            label,
            (x1, y1 - 10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (0, 0, 255),
            2,
        )

  current_time = time.time()

  # --- ३. Siren & Auto-Save Logic ---
  if pothole_detected and (current_time - last_alert_time > ALERT_INTERVAL):
    # १. सायरन अ‍ॅलर्ट प्ले करा
    play_siren()

    # २. स्क्रीनशॉट सेव्ह करा
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = os.path.join(SAVE_DIR, f"pothole_{timestamp}.jpg")
    cv2.imwrite(filename, frame)

    print(f"🚨 [ALERT] Pothole detected! Siren played & saved: {filename}")
    last_alert_time = current_time

  # --- ४. Display Overlay ---
  cv2.putText(
      frame,
      f"Potholes Detected: {pothole_count}",
      (30, 50),
      cv2.FONT_HERSHEY_SIMPLEX,
      1,
      (0, 255, 255),
      2,
  )

  if pothole_detected:
    cv2.putText(
        frame,
        "🚨 SIREN ALERT!",
        (30, 90),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (0, 0, 255),
        2,
    )

  cv2.imshow("Pothole Detection with Siren Alert", frame)

  if cv2.waitKey(1) & 0xFF == ord("q"):
    break

cap.release()
cv2.destroyAllWindows()

[INFO] Loading YOLO Model...
[INFO] Opening Video Stream...
[INFO] System Active. Press 'q' to quit.
